# Tech Challenge Fase 2
## 03.5 — Gold Power BI

Prepara uma base consolidada, estável e pronta para consumo pelo Power BI.

## 1. Imports

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(config["environment"]["base_path"])
SILVER_PATH = Path(config["paths"]["silver_path"])
GOLD_PATH = Path(config["paths"]["gold_path"])
LOG_PATH = Path(config["paths"]["log_path"])
CONFIG_PATH = Path(config["paths"]["config_path"])
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def ler_csv(caminho, sep=";", decimal=","):
    return pd.read_csv(
        caminho,
        sep=sep,
        decimal=decimal,
        encoding="utf-8",
        low_memory=False
    )


def converter_numero(serie):
    return (
        serie
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "<NA>": np.nan,
            "-": np.nan
        })
        .pipe(pd.to_numeric, errors="coerce")
    )


def normalizar_codigo(serie):
    return (
        serie
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.strip()
    )


def salvar_csv(df, destino, nome_arquivo):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

## 4. Preparação da base Power BI

In [0]:
df_powerbi = ler_csv(
    GOLD_PATH / "indicadores" / "GOLD_INDICADORES.csv"
)

colunas_prioritarias = [
    "ano",
    "sigla_uf",
    "id_municipio",
    "nome_municipio",
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "META_FINAL_2030",
    "gap_meta_2030",
    "risco_educacional",
    "qtd_alunos",
    "qtd_escolas",
    "qtd_presentes_lp",
    "qtd_alfabetizados",
    "taxa_participacao_lp",
    "taxa_alfabetizacao_alunos",
    "proficiencia_media_lp"
]

colunas_disponiveis = [
    c for c in colunas_prioritarias
    if c in df_powerbi.columns
]

df_powerbi = df_powerbi[colunas_disponiveis].copy()

display(df_powerbi.head())

## 5. Persistência

In [0]:
salvar_csv(
    df_powerbi,
    GOLD_PATH / "exports_powerbi",
    "POWERBI_BASE_CONSOLIDADA.csv"
)

print("Base Power BI salva com sucesso.")